# Самостоятельная работа: языковая модель на Elman RNN

В этом ноутбуке требуется самостоятельно собрать словарную языковую
модель следующего слова. Учебный маршрут совпадает с подробным
примером, но все содержательные части реализации отмечены только
внутри кодовых ячеек «TODO(student)». Здесь используется отдельный
корпус русской классической литературы RuHeritage-Corpus, поэтому
данные, жанр и словарь не совпадают с примером.

## Маршрут самостоятельной работы

Выполняйте ячейки по порядку. В каждой спецификации указаны доступные
переменные, входы, ожидаемые формы и контрольная точка. Не вставляйте
готовое решение из примера: воспроизведите необходимые преобразования
самостоятельно. Индексы токенов являются входом модели, а целевые
индексы следующего токена — вторым аргументом функции потерь. Меток
классов документа в этой работе нет. Каждый этап имеет одну и ту же
структуру: цель, входы, действия, ожидаемый результат и контрольная
точка. Если проверка не пройдена, не переходите к следующему этапу.

## Этап 1. Настройка самостоятельного эксперимента

**Цель:** зафиксировать независимую от примера постановку. Константы
выбирают RuHeritage-Corpus, его каталог данных «data», закреплённую
ревизию и поле «text». Первые 1 200 документов этого *другого*
потока достаточны для учебного CPU-эксперимента.

**Результат этапа:** определены все размеры модели и специальные
индексы. **Проверьте:** PAD, UNK, BOS и EOS равны соответственно
0, 1, 2 и 3; «MAX_SEQUENCE_LENGTH» равна 32. Эти числа — часть
контракта следующих этапов: индекс 0 будет маскироваться в потере, а
размерности 128 и два рекуррентных слоя определяют формы параметров.

In [ ]:
from __future__ import annotations

import copy
import math
import random
import re
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from datasets import load_dataset
from torch import nn
from torch.nn import functional as F
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

SEED = 42
DATASET_NAME = 'maxzt/RuHeritage-Corpus'
DATASET_REVISION = 'b2d01eaad3ddb119fe10c4cb17a86e47415a77cb'
DATASET_DATA_DIR = 'data'
TEXT_FIELD = 'text'
START_DOCUMENT = 0
MAX_DOCUMENTS = 1_200
MAX_WORDS_PER_DOCUMENT = 160
MAX_ORDINARY_WORDS = 6_000
MAX_SEQUENCE_LENGTH = 32
EMBEDDING_DIM = 128
HIDDEN_DIM = 128
NUM_LAYERS = 2
BATCH_SIZE = 64
LEARNING_RATE = 2e-3
WEIGHT_DECAY = 1e-4
MAX_EPOCHS = 12
PATIENCE = 3

SPECIAL_TOKENS = ['<pad>', '<unk>', '<bos>', '<eos>']
PAD_IDX, UNK_IDX, BOS_IDX, EOS_IDX = range(4)
DEVICE = torch.device('cpu')

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

## Этап 2. Получение литературных документов и нормализация

**Цель:** превратить поток записей литературного корпуса в два списка
коротких последовательностей слов. **Входы:** «DATASET_NAME»,
«DATASET_REVISION», «DATASET_DATA_DIR», «TEXT_FIELD» и строка поля
«text». При загрузке передайте каталог данных «data», закреплённую
ревизию, «split='train'» и потоковый режим.

**Действие:** прочитайте только первые 1 200 документов, нормализуйте
текст, выделите русские словарные токены и оставьте не более 160 слов
на документ. Каждый шестой непустой документ направьте в валидацию.

Направляйте каждый шестой непустой документ в валидацию, остальные —
в обучение. Результат должен состоять из двух непустых списков
документов, где каждый документ является списком строк. Выведите
размеры и пример токенов. **Математический смысл:** в этот момент
строки превращаются в последовательности дискретных символов, но
ещё не в числа и не в векторы. **Контрольная точка:** не скачивается
весь корпус; обе части непусты; используется поле «text» именно
RuHeritage-Corpus. Если документ после нормализации слишком короток
для пары «вход — следующий токен», его разумно исключить до разбиения
на окна.

In [ ]:
# Регулярное выражение: только русские буквы и дефис внутри слова
WORD_RE = re.compile(r"[а-яё]+(?:-[а-яё]+)*", re.IGNORECASE)

def normalize_and_tokenize(text: str, max_words: int = MAX_WORDS_PER_DOCUMENT) -> list[str]:
    text = text.lower()
    tokens = WORD_RE.findall(text)
    return tokens[:max_words]

# Потоковая загрузка
ds = load_dataset(
    DATASET_NAME,
    data_dir=DATASET_DATA_DIR,
    revision=DATASET_REVISION,
    split="train",
    streaming=True,
)

train_documents: list[list[str]] = []
val_documents: list[list[str]] = []

nonempty_index = 0
for i, record in enumerate(ds):
    if i >= START_DOCUMENT + MAX_DOCUMENTS:
        break
    if i < START_DOCUMENT:
        continue
    raw = record.get(TEXT_FIELD, "") or ""
    tokens = normalize_and_tokenize(raw)
    # Минимум 3 слова: bos + 1 токен + eos → хотя бы одна пара
    if len(tokens) < 3:
        continue
    if nonempty_index % 6 == 5:
        val_documents.append(tokens)
    else:
        train_documents.append(tokens)
    nonempty_index += 1

print("train docs:", len(train_documents), "val docs:", len(val_documents))
print("пример:", train_documents[0][:12] if train_documents else "—")
print("пример val:", val_documents[0][:12] if val_documents else "—")
assert train_documents and val_documents

## Этап 3. Построение словаря и пар обучения

**Цель:** получить конечное индексное представление текста и
сформировать учебные примеры. **Доступные данные:** «train_documents», «val_documents»,
«SPECIAL_TOKENS», «MAX_ORDINARY_WORDS», «MAX_SEQUENCE_LENGTH».
**Требуется:** построить словарь только по обучающим документам с
резервированием индексов 0–3; незнакомые слова в обеих частях должны
переходить в «<unk>».

**Действие:** затем добавьте «<bos>» и «<eos>», разбейте длинный документ на окна не
длиннее 33 индексов и создайте пары «вход — цель» сдвигом на один
токен. Один объект Dataset должен вернуть два одномерных тензора
одинаковой длины, не превышающей 32. Контрольная точка: для пары
«[bos, a, b, eos]» вход равен «[bos, a, b]», а цель — «[a, b, eos]».
**Математический смысл:** второй массив — не метка документа, а
правильный класс следующего слова на каждой позиции. **Формы:** до
объединения в мини-пакет обе части одной пары одномерны, $(L)$, где
$1\leq L\leq32$; только функция объединения добавит ось $B$.

In [ ]:
# Считаем частоты только по train
counter = Counter()
for doc in train_documents:
    counter.update(doc)

# Спецтокены + самые частые обычные слова
most_common = [w for w, _ in counter.most_common(MAX_ORDINARY_WORDS)]

itos = list(SPECIAL_TOKENS) + most_common
stoi = {w: i for i, w in enumerate(itos)}
VOCAB_SIZE = len(itos)

print("VOCAB_SIZE =", VOCAB_SIZE)

def encode_document(tokens: list[str]) -> list[int]:
    # bos + токены + eos
    return [BOS_IDX] + [stoi.get(w, UNK_IDX) for w in tokens] + [EOS_IDX]

def make_pairs(documents: list[list[str]]) -> list[tuple[list[int], list[int]]]:
    pairs = []
    window = MAX_SEQUENCE_LENGTH + 1  # 33
    for doc in documents:
        ids = encode_document(doc)
        for start in range(0, len(ids) - 1, MAX_SEQUENCE_LENGTH):
            chunk = ids[start:start + window]
            if len(chunk) < 2:
                continue
            x = chunk[:-1]  # ≤ 32
            y = chunk[1:]
            if len(x) == 0:
                continue
            pairs.append((x, y))
    return pairs

train_pairs = make_pairs(train_documents)
val_pairs = make_pairs(val_documents)

print("train pairs:", len(train_pairs), "val pairs:", len(val_pairs))
# контрольная точка: [bos, a, b, eos] → x=[bos,a,b], y=[a,b,eos]
sample_x, sample_y = train_pairs[0]
print("x:", sample_x[:8], "y:", sample_y[:8])
print("len(x) == len(y):", len(sample_x) == len(sample_y))

## Этап 4. Мини-пакет и маска технического заполнения

**Цель:** сделать пары одинаковой формы внутри одного мини-пакета.
**Вход:** список пар переменной длины. **Требуется:** написать Dataset
и функцию «collate_fn», которые вернут «input_ids» и «target_ids»
формы $(B,T)$ с $T\leq32$. Короткие строки дополните индексом
«PAD_IDX=0» справа. Создайте обучающий и валидационный DataLoader;
у обучающего допустимо перемешивание, у валидационного — нет.

**Результат:** первая полученная пара тензоров имеет одинаковую
двумерную форму, а заполнение присутствует только справа. Не путайте
«target_ids» с метками классов: это правильные следующие слова.
**Контрольная точка:** число позиций, где «target_ids» не равно
«PAD_IDX», положительно; именно эти позиции позднее войдут в среднюю
кросс-энтропию.

In [ ]:
class LanguageModelDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        x, y = self.pairs[idx]
        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)

def collate_language_batch(batch):
    xs, ys = zip(*batch)
    # PAD_IDX = 0, padding справа
    input_ids = pad_sequence(xs, batch_first=True, padding_value=PAD_IDX)
    target_ids = pad_sequence(ys, batch_first=True, padding_value=PAD_IDX)
    return input_ids, target_ids

train_loader = DataLoader(
    LanguageModelDataset(train_pairs),
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_language_batch,
)
val_loader = DataLoader(
    LanguageModelDataset(val_pairs),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_language_batch,
)

input_ids_demo, target_ids_demo = next(iter(train_loader))
print("input_ids:", input_ids_demo.shape, "target_ids:", target_ids_demo.shape)
print("T ≤ 32:", input_ids_demo.shape[1] <= MAX_SEQUENCE_LENGTH)
print("ненулевые цели:", int((target_ids_demo != PAD_IDX).sum().item()))

## Этап 5. Ручная рекуррентная сеть и прогноз логитов

**Цель:** реализовать отображение индексов слов в оценки следующего
слова. **Вход модели:** «input_ids» формы $(B,T)$ со значениями от 0 до
$V-1$. **Требуемая архитектура:**
$(B,T)\rightarrow\operatorname{Embedding}(V,128)\rightarrow
\operatorname{ElmanRNN}(128,2)\rightarrow\operatorname{Linear}(128,V)$.

**Действие:** реализуйте собственный рекуррентный блок с обучаемыми входными и
рекуррентными весами, смещением и функцией tanh. Затем реализуйте два
последовательных слоя и выходную линейную проекцию. Не используйте
готовый «nn.RNN». Контрольная точка одного прямого прохода:
эмбеддинги и состояния имеют форму $(B,T,128)$, логиты — $(B,T,V)$.
Логиты ещё не являются вероятностями: это $V$ вещественных оценок,
к которым функция потерь внутренне применит softmax.

**Порядок вычислений:** индекс выбирает строку таблицы эмбеддингов;
на каждом времени рекуррентный блок объединяет этот вектор с прежним
состоянием; верхний слой получает результат нижнего; линейный слой
создаёт оценки всех слов. Результат модели обязан зависеть от
параметров класса, иначе «backward()» не сможет найти направление
их изменения.

In [ ]:
class ElmanRNNUnit(nn.Module):
    def __init__(self, input_size: int, hidden_size: int):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.W_x = nn.Parameter(torch.empty(input_size, hidden_size))
        self.W_h = nn.Parameter(torch.empty(hidden_size, hidden_size))
        self.b   = nn.Parameter(torch.zeros(hidden_size))
        # инициализация Xavier-подобная
        nn.init.xavier_uniform_(self.W_x)
        nn.init.xavier_uniform_(self.W_h)

    def forward(self, x_t: torch.Tensor, h_previous: torch.Tensor) -> torch.Tensor:
        return torch.tanh(x_t @ self.W_x + h_previous @ self.W_h + self.b)


class ElmanRNN(nn.Module):
    def __init__(self, hidden_size: int, num_layers: int):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.layers = nn.ModuleList(
            [ElmanRNNUnit(hidden_size, hidden_size) for _ in range(num_layers)]
        )

    def forward(self, embeddings: torch.Tensor) -> torch.Tensor:
        B, T, H = embeddings.shape
        h_states = [torch.zeros(B, H, device=embeddings.device) for _ in range(self.num_layers)]
        outputs = []
        for t in range(T):
            x = embeddings[:, t, :]
            for li, layer in enumerate(self.layers):
                h_states[li] = layer(x, h_states[li])
                x = h_states[li]
            outputs.append(x)
        return torch.stack(outputs, dim=1)  # (B, T, H)


class ElmanLanguageModel(nn.Module):
    def __init__(self, vocab_size: int):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, EMBEDDING_DIM, padding_idx=PAD_IDX)
        self.rnn = ElmanRNN(HIDDEN_DIM, NUM_LAYERS)
        self.head = nn.Linear(HIDDEN_DIM, vocab_size)

    def forward(self, input_ids: torch.Tensor):
        emb = self.embedding(input_ids)          # (B,T,128)
        h = self.rnn(emb)                        # (B,T,128)
        logits = self.head(h)                    # (B,T,V)
        return logits

# Проверка одного прямого прохода
model = ElmanLanguageModel(VOCAB_SIZE).to(DEVICE)
logits_demo = model(input_ids_demo.to(DEVICE))
print("logits:", logits_demo.shape, "V =", VOCAB_SIZE)

## Этап 6. Обучение по сдвинутым целям и ранняя остановка

**Цель:** изменить параметры так, чтобы правильный следующий токен
получал больший логит. **Входы каждого шага:** «input_ids» и «target_ids» формы $(B,T)$.
**Требуется:** получить логиты $(B,T,V)$, развернуть логиты и цели по
позициям, создать «CrossEntropyLoss(ignore_index=PAD_IDX)» и AdamW.
Усредняйте потерю только по незаполненным позициям; сохраняйте лучшее
состояние по валидационной потере; завершайте обучение после трёх эпох
без улучшения, но не позже 12-й.

**Порядок вычислений:** в обучающем цикле очистите градиенты, выполните прямой проход,
«backward()» и «step()». На валидации обновление параметров запрещено.
Храните списки «history['loss']» и «history['val_loss']». Контрольная
точка: все значения конечны, а потеря обучения в целом снижается.
При вычислении итоговой эпохи суммируйте отрицательные логарифмы и
число реальных токенов по всем мини-пакетам отдельно: среднее средних
при разных длинах мини-пакетов дало бы смещённую оценку.

In [ ]:
model = ElmanLanguageModel(VOCAB_SIZE).to(DEVICE)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX, reduction="sum")
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

history = {"loss": [], "val_loss": []}
best_val = float("inf")
best_state = None
epochs_without_improvement = 0

def run_epoch(loader, train: bool):
    model.train(train)
    total_nll = 0.0
    total_tokens = 0
    for input_ids_b, target_ids_b in loader:
        input_ids_b = input_ids_b.to(DEVICE)
        target_ids_b = target_ids_b.to(DEVICE)

        if train:
            optimizer.zero_grad()

        logits = model(input_ids_b)                # (B,T,V)
        B, T, V = logits.shape
        loss = criterion(logits.reshape(B * T, V), target_ids_b.reshape(B * T))

        if train:
            loss.backward()
            optimizer.step()

        n_tokens = int((target_ids_b != PAD_IDX).sum().item())
        total_nll += float(loss.item())
        total_tokens += n_tokens

    return total_nll / max(total_tokens, 1)

for epoch in range(MAX_EPOCHS):
    train_loss = run_epoch(train_loader, train=True)
    val_loss = run_epoch(val_loader, train=False)
    history["loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    print(f"epoch {epoch+1:02d} | train={train_loss:.4f} | val={val_loss:.4f} | ppl={math.exp(val_loss):.2f}")

    if val_loss < best_val - 1e-5:
        best_val = val_loss
        best_state = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print("early stop at epoch", epoch + 1)
            break

if best_state is not None:
    model.load_state_dict(best_state)

## Этап 7. Оценка, перплексия и осмысленная интерпретация

**Цель:** проверить модель на данных, по которым не происходило
обновление параметров. **Вход:** лучшая обученная модель и валидационный DataLoader.
**Требуется:** вычислить среднюю кросс-энтропию по токенам и
перплексию $\exp(\mathcal L)$, построить графики «loss» и «val_loss»,
получить top-$k$ продолжений короткого русского запроса и выполнить
жадную генерацию до EOS или лимита.

Контрольная точка: перплексия — конечное положительное число; top-$k$
содержит слова из «itos»; цикл генерации имеет явный предел шагов.
В отчёте объясните, почему маленькая модель может выдавать частые, но
не всегда содержательные продолжения. Сопоставьте каждый top-$k$
прогноз с тем, что логиты последней позиции сначала превращаются в
распределение softmax, а затем сортируются по вероятности.

In [ ]:
# 1. Оценка
val_loss = run_epoch(val_loader, train=False)
val_ppl = math.exp(val_loss)
print(f"val loss = {val_loss:.4f}, perplexity = {val_ppl:.3f}")

# 2. Графики
plt.figure(figsize=(7, 4))
plt.plot(history["loss"], label="train")
plt.plot(history["val_loss"], label="val")
plt.xlabel("epoch"); plt.ylabel("loss")
plt.legend(); plt.grid(True); plt.show()

# 3. top-k продолжений
model.eval()

def predict_top_k(prompt: str, k: int = 10):
    tokens = normalize_and_tokenize(prompt, max_words=MAX_SEQUENCE_LENGTH)
    ids = [BOS_IDX] + [stoi.get(w, UNK_IDX) for w in tokens]
    ids = ids[-(MAX_SEQUENCE_LENGTH):]  # короче T
    x = torch.tensor([ids], dtype=torch.long, device=DEVICE)
    with torch.no_grad():
        logits = model(x)                 # (1, T, V)
    last = logits[0, -1, :]               # (V,)
    probs = F.softmax(last, dim=-1)
    top_probs, top_ids = probs.topk(k)
    return [(itos[i], float(p)) for i, p in zip(top_ids.tolist(), top_probs.tolist())]

prompt = "в начале"
print("prompt:", prompt)
for w, p in predict_top_k(prompt, k=10):
    print(f"  {w!r:>16}  p={p:.4f}")

# 4. Жадная генерация
def greedy_generate(prompt: str, max_new_tokens: int = 20):
    tokens = normalize_and_tokenize(prompt, max_words=MAX_SEQUENCE_LENGTH)
    ids = [BOS_IDX] + [stoi.get(w, UNK_IDX) for w in tokens]
    ids = ids[-(MAX_SEQUENCE_LENGTH - max_new_tokens):]
    generated = list(tokens)
    for _ in range(max_new_tokens):
        x = torch.tensor([ids], dtype=torch.long, device=DEVICE)
        with torch.no_grad():
            logits = model(x)
        next_id = int(logits[0, -1].argmax().item())
        if next_id == EOS_IDX:
            break
        if next_id in (PAD_IDX, BOS_IDX):
            # избегаем служебных токенов в выводе
            probs = F.softmax(logits[0, -1], dim=-1)
            probs[PAD_IDX] = 0; probs[BOS_IDX] = 0
            next_id = int(probs.argmax().item())
            if next_id == EOS_IDX:
                break
        generated.append(itos[next_id])
        ids.append(next_id)
        if len(ids) > MAX_SEQUENCE_LENGTH:
            ids = ids[-MAX_SEQUENCE_LENGTH:]
    return " ".join(generated)

print(greedy_generate("в начале", max_new_tokens=20))

## Этап 8. Самопроверка перед сдачей

- Использован RuHeritage-Corpus, его поле «text», каталог данных
  «data» и закреплённая ревизия.
- Словарь построен только по обучающей части; индексы PAD, UNK, BOS,
  EOS равны 0, 1, 2, 3.
- «input_ids» и «target_ids» имеют форму $(B,T)$, $T\leq32$.
- Эмбеддинги и скрытые состояния имеют форму $(B,T,128)$; логиты —
  $(B,T,V)$.
- PAD исключён из кросс-энтропии, а не заменён на обычную цель.
- Потери и перплексия конечны; сохранена лучшая валидационная модель.
- В коде нет TensorFlow, Keras и готового «nn.RNN».